In [3]:
"""
Statistical Analysis Toolkit

RULE: implement every calculation yourself with plain Python.
NumPy may be imported ONLY in compare_with_numpy(), to verify your results.
Do not use the statistics module.
"""
import csv
import math
from pathlib import Path

DATA_PATH = Path("data") / "agent_deposits.csv"

---

## cleaning data

In [26]:
def load_column(path: Path, column: str) -> tuple[list[float], list[tuple[int, str]]]:
    """Return (valid_values, rejected) where rejected holds (row_number, raw_value) pairs."""
    valid_values, rejected = [], []

    # importing data using dict reader
    with open(path, newline="") as f:
        for index, row in enumerate(csv.DictReader(f), start=2):
            raw_data = row[column]
            try:
                # attmept to convert col to float
                valid_values.append(float(raw_data))
            except ValueError:
                # if float conversion fails, find reason 
                reason = "missing value" if raw_data.strip() == "" else "Not a number"
                rejected.append((index, raw_data, reason)) 

    return valid_values, rejected


# viewing unusable data in daily deposit and transaction cols
valid_deposits, rejected_deposits = load_column(DATA_PATH, "daily_deposits_kes")

print(rejected_deposits)

valid_txns, rejected_txns = load_column(DATA_PATH, "transactions")
print(rejected_txns)

# print(valid_deposits)
# 6 columns unuseable

[(7, '', 'missing value'), (10, '-', 'Not a number'), (14, 'n/a', 'Not a number'), (25, '12,40O', 'Not a number'), (29, '', 'missing value')]
[(7, '', 'missing value'), (10, '', 'missing value'), (14, '', 'missing value'), (17, 'abc', 'Not a number'), (29, '', 'missing value')]


---

In [8]:
def mean(values: list[float]) -> float:
    return sum(values) / len(values)


# finding mean of valid deposits
deposits_mean = mean(valid_deposits)
print("Mean of valid deposits")
print(f"{deposits_mean:,.2f}")
print()

# finding mean of valid transactions
print("Mean of vlaid txns")
txns_mean = mean(valid_txns)
print(f"{txns_mean:,.2f}")

Mean of valid deposits
22,400.00

Mean of vlaid txns
93.11


In [ ]:
def median(values: list[float]) -> float:
    # order values using sorted function
    ordered = sorted(values)
    # if len of list is even 
    mid = len(ordered) // 2
    # if len of list is odd,
    if len(ordered) % 2 == 1:
        # return value at mid
        return ordered[mid]
    # find median by avg of center values
    return (ordered[mid -1] + ordered[mid]) / 2


# finding median of valid deposits
deposits_median = median(valid_deposits)
print("Deposits median")
print(deposits_median)
print()

# finding meadian of valid txns
print("Txns median")
txns_median = median(valid_txns)
print(txns_median)

Deposits median
18805.47

Txns median
94.0


In [21]:
def mode(values: list[float]) -> list[float]:
    """Return every value that occurs most often (a list, because there can be ties or none)."""
    # initialize empty set counts to hold unique values
    counts = {}

    # count every time a value is found, start count at 0, add 1 for each find
    for value in values:
        counts[value] = counts.get(value, 0) + 1

    # if no value is found, no mode, return empty list
    if not counts:
        return []

    # find highest count
    highest = max(counts.values())
    # if all values only appear once, no mode, return empty list
    if highest == 1:
        return []

    # return value if it has the highest count
    return [value for value, count in counts.items() if count == highest]


# finding mode
# finding txns mode
txns_mode = mode(valid_txns)
print("Txns Mode")
print(txns_mode)



Txns Mode
[132.0, 73.0]


In [27]:
def variance(values: list[float], sample: bool = True) -> float:
    """Sample variance (n - 1) by default; population variance (n) when sample is False."""
    # finding count of values passed
    n = len(values)
    # if < 2 values
    if n < 2:
        raise ValueError("Need at least 2 values to calculate vaariance!")
    data_mean = mean(values)
    # find diff of value - mean
    # square diff, to make it posive
    # sum all +ve squared values
    squared_diffs = sum((v- data_mean) ** 2 for v in values)
    # divide sum by count - 1
    return squared_diffs / (n - 1 if sample else n)


# test variance
deposits_var = variance(valid_deposits)
print("Deposits Variance")
print(deposits_var)
print()

txns_var = variance(valid_txns)
print("Txns variance")
print(txns_var)

Deposits Variance
328019873.35939974

Txns variance
502.5747899159664


In [25]:
def std_dev(values: list[float], sample: bool = True) -> float:
    # std dev id square root of variance so use power 0.5 to find square root
    return variance(values, sample) ** 0.5


# test std dev
deposits_std_dev = std_dev(valid_deposits)
print("Deposits std dev")
print(deposits_std_dev)
print()

txns_std_dev = std_dev(valid_txns)
print("Txns std dev")
print(txns_std_dev)

Deposits std dev
18111.318929315992

Txns std dev
22.418179897484237


In [ ]:
def quartiles(values: list[float]) -> tuple[float, float, float]:
    """Return (Q1, Q2, Q3). Document which quartile method you chose."""
    # using inclusive method, Return (Q1, Q2, Q3) using linear interpolation between closest ranks
    
    # if no values passed
    if len(values) < 2:
        raise ValueError("Need more than 2 values to find quartiles")

    # order data in acsending order
    s = sorted(values)
    # finding mid point
    half = len(s) // 2
    # slicing data at midpoint for lower and upper segment
    lower, upper = s[:half], s[len(s) -half:]
    # using median to find quartiles for lower, midpoint and upper 
    return median(lower), median(s), median(upper)


# testing quartiles
deposits_quats = quartiles(valid_deposits)
print("deposits quartiles")
print(deposits_quats)
print()

txns_quats = quartiles(valid_txns)
print("txns quartiles")
print(txns_quats)
print()

deposits quartiles
(15988.21, 18805.47, 21195.44)

deposits quartiles
(73.0, 94.0, 110.0)



In [32]:
def iqr_outliers(values: list[float], k: float = 1.5) -> dict:
    """Return the IQR, the lower and upper fences, and the values outside them."""
    q1, _, q3 = quartiles(values)
    iqr = q3 - q1
    lower_limit = q1 - k * iqr
    upper_limit = q3 + k * iqr

    return {
        "iqr" : iqr,
        "lower_limit" : lower_limit,
        "upper_limit": upper_limit,
        # return value from values
        "outliers" : [v for v in values 
                      # if value < or > than limits
                      if v < lower_limit 
                      or v > upper_limit],
    }



# testing IQR
deposits_IQR = iqr_outliers(valid_deposits)
print("deposits outliers")
print(deposits_IQR)
print()

txns_IQR = iqr_outliers(valid_txns)
print("txns quartiles")
print(txns_IQR)
print()

deposits outliers
{'iqr': 5207.23, 'lower_limit': 8177.365, 'upper_limit': 29006.284999999996, 'outliers': [96500.0, 1200.0, 88400.0, 29060.99]}

txns quartiles
{'iqr': 37.0, 'lower_limit': 17.5, 'upper_limit': 165.5, 'outliers': []}



In [ ]:


class StatisticsAnalyzer:
    """Wraps one numeric column and exposes every statistic above as a method."""

    def __init__(self, values: list[float], name: str = "values") -> None:
        raise NotImplementedError

    @classmethod
    def from_csv(cls, path: Path, column: str) -> "StatisticsAnalyzer":
        raise NotImplementedError

    def summary(self) -> dict:
        raise NotImplementedError

    def without_outliers(self) -> "StatisticsAnalyzer":
        raise NotImplementedError

    def report(self) -> str:
        raise NotImplementedError


def compare_with_numpy(analyzer: StatisticsAnalyzer) -> dict[str, bool]:
    """Import numpy HERE, recompute each statistic and return {name: matches}."""
    raise NotImplementedError


if __name__ == "__main__":
    # TODO
    pass
